In [1]:
%load_ext autoreload
%autoreload 2

from tensorboard.backend.event_processing import event_accumulator
import os
import scipy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import copy
from tqdm import tqdm
import csv
from scipy.special import softmax
device = torch.device('cuda:0')
import random
import pickle
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(filename='train_log.log', level=logging.INFO)

from Dataset import *
from GAN import GAN, WGAN_GP
from Discriminator import DataDiscriminator, DeepSetCritic
from util import set_seed, jsd_from_samples
import itertools
import pandas as pd
#optimize over w directly, dont use any theta shenannigans. 
from sklearn import preprocessing
import pytorch_warmup as warmup
from DataProcessing import *
from attribution import *

In [ ]:
NUM_TRIALS = 1

BATCH_SIZE = 2
SUBSET_SIZE = 32
GENERATOR_TRAINING_FACTOR = 1
DISCRIMINATOR_TRAINING_FACTOR = 8
GT_LIMIT = 25000
BIAS_LIMIT = 2500
LAMBDAGP = 0
LAMBDAW = 1 #40
LAMBDAD = 1
LAMBDA_FIRST_LAYER = 1
GENERATOR_LEARNING_RATE = 1e-5
DISCRIMINATOR_LEARNING_RATE = 1e-4
BATCHS_IN_EPOCH = 1
EPOCHS = 300 # the stream is infinite so one epoch will be defined as BATCHS_IN_EPOCH * BATCH_SIZE

GEN_HISTORY_LENGTH = 0
WARMUP_EPOCHS = 300
SAVE_DATASET = False
SAVE_WEIGHTS = True
GEN_LAYERS = [1024, 1024, 1024] #[256 for _ in range(5)]
DISC_LAYERS = [1024, 1024, 1024]
GEN_DROPOUT = 0.2
DISC_DROPOUT = 0.2
TEMPERATURE_START = 1
TEMPERATURE_END = 0.3
TEMPERATURE = 0.1
SAME_DATA_GT = False
SAME_DATA_BIAS = False
SAME_DATA_SEEN = False
SAME_NETWORK_INIT = False

fixed_seed = [np.random.randint(1e6)]
all_rngs = []
print('SETTING SEED: ', fixed_seed)
if type(fixed_seed) == list:
    for _ in range(NUM_TRIALS):
        t_rng = []
        for seed in fixed_seed:
            t_rng.append(set_seed(seed,
                                    device,
                                data_init=[SAME_DATA_GT, SAME_DATA_BIAS],
                                data_gen =SAME_DATA_SEEN,
                                network_init=SAME_NETWORK_INIT,))
        all_rngs.append(t_rng)
else:
    for _ in range(NUM_TRIALS):
        all_rngs.append(set_seed(fixed_seed,
                                device,
                            data_init=[SAME_DATA_GT, SAME_DATA_BIAS],
                            data_gen =SAME_DATA_SEEN,
                            network_init=SAME_NETWORK_INIT,))
all_names = ['REGION', 'EDUC', 'INCTOT', 'SEX', 'MARST', 'RACE', 'AGE', 'rand1', 'rand2', 'rand3', 'rand4', 'rand5', 'rand6', 'rand7']
all_results_dict = {}
var_index_to_names = {}
names_to_idx = {}
for i, name in enumerate(all_names):
    var_index_to_names[i] = name
    all_results_dict[name] = [[],[]]
    names_to_idx[name] = i

for i in range(NUM_TRIALS):
    rngs = all_rngs[i]
    d = HouseholdPulse_synthetic(ground_truth_path='./data/censusHouseholdPulse_data/cleaned/ipums_cleaned.csv',
                                        bias_path = './data/censusHouseholdPulse_data/cleaned/pulse_week29_cleaned.csv',
                                        rngs=rngs[0],
                                        device=device,
                                        gt_limit = GT_LIMIT*len(rngs),
                                        bias_limit = BIAS_LIMIT, 
                                        )
    for cdx in range(d.unscaled_biased.shape[1]):
        name = var_index_to_names[cdx]
        jsd_gap = jsd_from_samples(d.unscaled_ground_truth[:,cdx],d.unscaled_biased[:,cdx])
        assert name == var_index_to_names[cdx]
        if name in d.column_names:
            all_results_dict[name][0].append(jsd_gap)
        else:
            all_results_dict[name][1].append(jsd_gap)

In [3]:
#code for loading and  analyzing synth runs with top contributing vars
'''
1. Average gradient list over 100 runs compared to runs
2. Point system based on top 5 placement in gradients

3. (1-2) for points past 200
'''
import ast

def compute_sens_spec(STV_dict, RV_dict):
    '''
    STV_dict = point dict for Selected True Variables
    RV_dict = point dict for random variables

    specificty = TP / ( TP + FN )
    sensitivty = TN / (TN + FP)
    '''
    sens_num = 0
    sens_denom = 0
    spec_num = 0
    spec_denom = 0
    for k, v in STV_dict.items():
        sens_num += v[0]
        sens_denom += v[0]

        spec_denom += v[1] - v[0]
    
    for k, v in RV_dict.items():
        sens_denom += v[0]

        spec_num += v[1]
        spec_denom += v[1]

    print(sens_denom, spec_denom)
    return sens_num/sens_denom, spec_num / spec_denom

def compute_mean_grad(history,embedding_dict,
                      start_point=0,end_point=0):
    history = np.abs(history)
    all_grads = []
    summer = 0
    for k,v in embedding_dict.items():
        summer += v[2]

    for cgrad_idx in range(start_point,end_point):
        all_grads.append(process_grad(history[cgrad_idx],embedding_dict))

    lines = [[] for _ in range(len(all_grads[0]))]
    for g in all_grads:
        for var_idx in g:
            lines[var_idx].append(g[var_idx])
    
    lines = np.array(lines)
    return np.mean(lines,axis=1)

def normalize_x(values):
    normalized = [(x - min(values)) / (max(values) - min(values)) for x in values]
    return normalized

def calc_points(g_history, fname,
                embedding_dict, var_list, 
                points, rand_points, top_x,
                start, stop):
    mean_grads = compute_mean_grad(g_history,embedding_dict,
                                   start_point=start,
                                   end_point=stop)
    local_names_sorted, local_mean_grad_sorted = zip(*sorted(zip(copy.deepcopy(var_list), mean_grads), key=lambda x: x[1],reverse = True))
    for fname_k in fname:
        if fname_k in local_names_sorted[:top_x]:
            points[fname_k][0] += 1
        points[fname_k][1] += 1
    
    
    for i in range(1,8):
        rand_name = "rand_"+str(i)
        if rand_name in local_names_sorted[:top_x]:
            rand_points[rand_name][0] += 1
        else:
            rand_points[rand_name][1] += 1

def compute_means_rank(g_history, fname,
                embedding_dict, var_list, 
                points, rand_points, top_x,
                start, stop):

    ret_dict = {}
    mean_grads = compute_mean_grad(g_history,embedding_dict,
                                   start_point=start,
                                   end_point=stop)
    local_names_sorted, local_mean_grad_sorted = zip(*sorted(zip(copy.deepcopy(var_list), mean_grads), key=lambda x: x[1],reverse = True))
    #normalized = (local_mean_grad_sorted - np.min(local_mean_grad_sorted)) / (np.max(local_mean_grad_sorted) - np.min(local_mean_grad_sorted))

    return local_names_sorted, local_mean_grad_sorted

def create_rank_dict(all_names_sorted, all_means_sorted):
    ret_dict = {}
    for lns, lgs in zip(all_names_sorted, all_means_sorted):
        for i, ln in enumerate(lns):
            if ln not in ret_dict:
                ret_dict[ln] = [lgs[i]]
            else:
                ret_dict[ln].append(lgs[i])
    #mean of the means
    new_ret_dict = {}
    for k in ret_dict:
        new_ret_dict[k] = np.mean(ret_dict[k])
    mean_val = np.mean(list(new_ret_dict.values()))
    #then normalize
    final_ret_dict = {}
    for k in ret_dict:
        final_ret_dict[k] = new_ret_dict[k] / mean_val
    return final_ret_dict


def convert_point_dict_to_df(points,
                             col_names):
    sorted_v = []
    sorted_k = []
    sorted_tots = []
    for k, v in sorted(points.items(), key=lambda item: item[1][0], reverse = True):
        #print(k, v/(5*100))
        sorted_v.append(v[0])
        sorted_tots.append(v[1])
        sorted_k.append(k)
    df = pd.DataFrame([sorted_v,sorted_tots])
    df.columns = sorted_k
    df.index = col_names
    return df.T

#analyzing grad history}
file_path = "./saves_week29/"
all_files = os.listdir(file_path)

counter = 0
all_g_history = []
all_c_history = []
all_embedding_dicts = []
all_names = []

for fname in all_files:
    if "embedding" in fname or "runs" in fname:
        continue
    
    suffix_isolated = fname.split("history")[1]
    suffix_isolated = suffix_isolated.split(".")[0]
    with open(file_path+"embedding_dict"+suffix_isolated+".pikl", 'rb') as file:
        embedding_dict = pickle.load(file)
    all_embedding_dicts.append(embedding_dict)
    #fname2 = "grad_history_0_['INCTOT', 'SEX'].npz"
    #if fname != "grad_history_0_['SEX', 'AGE'].npz":
    #    continue
    
    path = file_path+fname

    #syn parsing
    if False:
        fname_isolated = fname.split("_")[3]
        fname_isolated = fname_isolated.split(".")[0]
        fname_isolated = ast.literal_eval(fname_isolated)
    else:
        fname_isolated = fname.split("_")
        print(fname_isolated)
        fname_isolated = fname_isolated[2]

    fname_isolated = list(dict.fromkeys(s for tup in fname_isolated for s in tup))
    all_names.append(fname_isolated)
    
    loaded = np.load(path)

    all_g_history.append(loaded['w'])
    all_c_history.append(loaded['wc'])
    
    if False:
        top_x = 2
        if True:
            gen_points = calc_points(g_history,
                                    fname=fname_isolated,
                                    embedding_dict=embedding_dict,
                                    var_list=var_list,
                                    points=all_g_points,
                                    rand_points=all_g_rand,
                                    top_x = top_x,
                                    start=start_point,
                                    stop=stop_point)
        if True:
            critic_points = calc_points(c_history,
                                        fname=fname_isolated,
                                        embedding_dict=embedding_dict,
                                        var_list=var_list,
                                        points=all_c_points,
                                        rand_points=all_c_rand,
                                        top_x = top_x,
                                        start=start_point,
                                        stop=stop_point)
        
        gen_points_df = convert_point_dict_to_df(all_g_points,
                                                col_names=["Times in Top K", "Total Times Selected"])
        all_gen_rand_df = convert_point_dict_to_df(all_g_rand,
                                                col_names = ["Times in Top K", "times outside of top K"])
        critic_point_df = convert_point_dict_to_df(all_c_points,
                                                col_names = ["Times in Top K", "Total Times Selected"])
        all_critic_rand_df = convert_point_dict_to_df(all_c_rand,
                                                    col_names = ["Times in Top K", "times outside of top K"])
        if False:
            print(mean_grads.shape)
            var_list =  ['REGION', 'EDUC', 'INCTOT', 'SEX', 'MARST', 'RACE', 'AGE']
            for i in range(mean_grads.shape[1]):
                if i <= 6:
                    plt.scatter(np.arange(mean_grads.shape[0]),mean_grads[:,i], label=var_list[i])
            plt.legend()
            plt.ylim(0, 0.10)
            plt.title(path)
            plt.show()
        if False:
            all_grads = np.array(all_grads)
            averaged_experiment_grad = np.mean(all_grads,axis=0)
            averaged_experiment_grad = list(averaged_experiment_grad)
            names_sorted, mean_grad_sorted = zip(*sorted(zip(var_list, averaged_experiment_grad), key=lambda x: x[1],reverse = True))
            #mean_grad_sorted /= np.max(mean_grad_sorted)
            #mean_grad_sorted = normalize_x(mean_grad_sorted)
            mean_grad_sorted = mean_grad_sorted

['grad', 'history', '0', '.npz']
['grad', 'history', '1', '.npz']
['grad', 'history', '2', '.npz']
['grad', 'history', '3', '.npz']
['grad', 'history', '4', '.npz']
['grad', 'history', '5', '.npz']
['grad', 'history', '6', '.npz']
['grad', 'history', '7', '.npz']
['grad', 'history', '8', '.npz']
['grad', 'history', '9', '.npz']
['grad', 'history', '10', '.npz']
['grad', 'history', '11', '.npz']
['grad', 'history', '12', '.npz']
['grad', 'history', '13', '.npz']
['grad', 'history', '14', '.npz']
['grad', 'history', '15', '.npz']
['grad', 'history', '16', '.npz']
['grad', 'history', '17', '.npz']
['grad', 'history', '18', '.npz']
['grad', 'history', '19', '.npz']
['grad', 'history', '20', '.npz']
['grad', 'history', '21', '.npz']
['grad', 'history', '22', '.npz']
['grad', 'history', '23', '.npz']
['grad', 'history', '24', '.npz']
['grad', 'history', '25', '.npz']
['grad', 'history', '26', '.npz']
['grad', 'history', '27', '.npz']
['grad', 'history', '28', '.npz']
['grad', 'history', '29'

In [4]:
print(all_g_history[0].shape)

(300, 2500, 33)


In [47]:
#run cell

for time_step in range(0,90,100):

    start_point = time_step
    stop_point = time_step+90
    print(start_point,stop_point)

    all_g_points = {}
    all_c_points = {}
    all_g_rand = {}
    all_c_rand = {}

    var_list =  ['REGION', 'EDUC', 'INCTOT', 'SEX', 'MARST', 'RACE', 'AGE']
    #for i in range(7):
    #    var_list.append("rand_"+str(i+1))
    all_grads = []
    all_g_points = {}
    all_c_points = {}
    all_g_rand = {}
    all_c_rand = {}
    for v in var_list:
        if "rand" not in v:
            all_g_points[v] = [0,0]
            all_c_points[v] = [0,0]
        else:
            all_g_rand[v] = [0,0]
            all_c_rand[v] = [0,0]
    
    all_names_sorted = []
    all_means_sorted = []
    for i in range(len(all_g_history)):
        print(i)
        names_sorted, means_sorted = compute_means_rank(all_g_history[i],
                    fname=all_names[i],
                    embedding_dict=all_embedding_dicts[i],
                    var_list=var_list,
                    points=all_g_points,
                    rand_points=all_g_rand,
                    top_x = len(all_names[i]),
                    start=start_point,
                    stop=stop_point)
        all_names_sorted.append(names_sorted)
        all_means_sorted.append(means_sorted)
    
    print(len(all_names_sorted))
    if len(all_names_sorted) > 0:
        ret_dict = create_rank_dict(all_names_sorted,all_means_sorted)
        print(ret_dict)
    else:
        gen_points_df = convert_point_dict_to_df(all_g_points,
                                                col_names=["Times in Top K", "Total Times Selected"])
        all_gen_rand_df = convert_point_dict_to_df(all_g_rand,
                                                col_names = ["Times in Top K", "times outside of top K"])

        #print("Generator TV")
        #print(gen_points_df)
        #print("Generator Random")
        #print(all_gen_rand_df)

        gen_sens, gens_spec = compute_sens_spec(all_g_points, all_g_rand)
        print("Sensitivity/Specificty Analysis:")
        print("Generator: ", gen_sens, ",", gens_spec)
        print("")

0 90
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
{'EDUC': 1.325914, 'INCTOT': 1.1722796, 'RACE': 1.131932, 'MARST': 0.96217066, 'AGE': 0.8607123, 'SEX': 0.74536157, 'REGION': 0.80162984}


In [48]:
for k, v in sorted(ret_dict.items(), key=lambda item: item[1]):
    print(k, v)

SEX 0.74536157
REGION 0.80162984
AGE 0.8607123
MARST 0.96217066
RACE 1.131932
INCTOT 1.1722796
EDUC 1.325914


In [ ]:
#d4p data processing

df = pd.read_stata("./data/progress_data/dfp_covid_tracking_poll.dta")
df = df[df['wave'] == 25]
columns_to_keep = []
columns_to_keep.append('nationalweight')
columns_to_keep.append('gender')
columns_to_keep.append('ethnicity')
columns_to_keep.append('education')
columns_to_keep.append('age')
columns_to_keep.append('region')
columns_to_keep.append('hhi') #household income
columns_to_keep.append('vax')

df = df[columns_to_keep].dropna(subset=columns_to_keep)


vax_binary = df['vax'].str.startswith('Yes,').astype(int)
weighted_avg = (vax_binary * df['nationalweight']).sum() / df['nationalweight'].sum()

#d4p procressing
df = pd.read_stata("./data/progress_data/dfp_covid_tracking_poll.dta")
df = df[df['wave'] == 25]

columns_to_keep = []
columns_to_keep.append('gender')
columns_to_keep.append('ethnicity')
columns_to_keep.append('education')
columns_to_keep.append('age')
columns_to_keep.append('region')
columns_to_keep.append('hhi') #household income
columns_to_keep.append('vax')

df = df[columns_to_keep].dropna(subset=columns_to_keep)

#d4p processing 2
def recode_gender(value):
    if value == 'Male':
        return 1
    elif value == 'Female':
        return 2
    else:
        return None
df['gender'] = df['gender'].apply(recode_gender)

def recode_census_age(value):
    if value >= 85: 
        return 5
    elif value >= 65:
        return 4
    elif value >= 50:
        return 3
    elif value >= 35:
        return 2
    elif value >= 18:
        return 1
df['age'] = df['age'].apply(recode_census_age)

def recode_census_region(value):
        if value == 1:
            return 1  # NorthEast
        elif value == 2:
            return 3  # MidWest
        elif value == 3:
            return 2  # South
        elif value == 4:
            return 4  # West
        else:
            return None  # Handle unexpected values
# Apply the recoding function to the region column in the census data
df['region'] = df['region'].apply(recode_census_region)

educ_mapping = {
            1: 1,  # Less than high school
            2: 2,  # High school graduate
            3: 3,  # Some college
            4: 3,  # Some college
            5: 3,  # Some college
            6: 4, # Bachelor's degree
            7: 5,  # Graduate degree
            8: 5,
            -3105: None,
        }
df['education'] = df['education'].map(educ_mapping)

race_map = {
        1: 1,  # White -> White, Alone
        2: 2,  # Black/African American -> Black, Alone
        3: 4,  # American Indian or Alaska Native -> Any other race alone, or race in combination
        5: 3,  # Chinese -> Asian, Alone
        7: 3,  # Japanese -> Asian, Alone
        4: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        6: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        8: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        9: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        10: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        11: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        12: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        13: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        14: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        15: None,  # Other race, nec -> Any other race alone, or race in combination
        16: None,
    }
df['ethnicity'] = df['ethnicity'].map(race_map)

def map_income(value):
    if value == -3015:
        return None  # Question seen but not selected
    elif value <= 3:
        return 1  # Less than $25,000
    elif 4 <= value < 6:
        return 2  # $25,000 - $34,999
    elif 6 <= value < 9:
        return 3  # $35,000 - $49,999
    elif 9 <= value < 14:
        return 4  # $50,000 - $74,999
    elif 14 <= value < 19:
        return 5  # $75,000 - $99,999
    elif 19 <= value < 21:
        return 6  # $100,000 - $149,999
    elif 21 <= value < 23:
        return 7  # $150,000 - $199,999
    else:
        return 8  # $200,000 and above

df['hhi'] = df['hhi'].apply(map_income)

df = df.rename(columns={'gender': 'SEX'})
df = df.rename(columns={'education': 'EDUC'})
df = df.rename(columns={'region': 'REGION'})
df = df.rename(columns={'hhi': 'INCTOT'})
df = df.rename(columns={'ethnicity': 'RACE'})
df = df.rename(columns={'age': 'AGE'})

vax_binary = df['vax'].str.startswith('Yes,').astype(int)
df.drop('vax',axis=1)
df['vax'] = vax_binary

cols = list(df.columns)
cols.insert(0, cols.pop(cols.index('vax')))
df = df[cols]

print(df)

week='25'
df.to_csv('./data/progress_data/week'+week+'_cleaned.csv', index=False)

In [ ]:
#d4p processing continued 
from Dataset import D4P_dataset
EPOCHS = 100
DISC_LR = 1e-5
GT_LIMIT = 100 #25000
BIAS_LIMIT = 100 #1000
BATCH_SIZE =16
SUBSET_SIZE = 64
week='25'

seed=0
rngs = set_seed(seed,
                device,
                data_init=[True,True],
                data_gen =True,
                network_init=True,)

D1_rngs = copy.deepcopy(rngs)
D1_rngs['seed_bias'] = 359556
D2_rngs = copy.deepcopy(rngs)
D2_rngs['seed_bias'] = 280651
#load in both data sets

d = D4P_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
            bias_path = './data/progress_data/week'+week+'_cleaned.csv',
            rngs=rngs,
            device=device,
            gt_limit = GT_LIMIT,
            )

In [ ]:
#analyzing the groupings of variables
#plotting events from saved run logs
# Path to the directory where SummaryWriter saved logs
import re
import ast
file_paths = ["runs_household_1_8_vars/"]

def isolate_variable_names(file_name):
    match = re.search(r"columns_to_keep:(\([^\)]*\))\|\|seed", file_name)
    if match:
        test = ast.literal_eval((match.group(1)))
        return test
    return None
def print_stats(runs_path):
    run_folders = []
    for name in os.listdir(runs_path):
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))

    all_vac_predictions = []
    all_l2_demographics = []
    all_names = []
    for i, event_file in enumerate(event_files):
        # Load event accumulator
        var_name_tuple = isolate_variable_names(event_file)
        if len(var_name_tuple) != num_vars:
            continue
        all_names.append(var_name_tuple)
        ea = event_accumulator.EventAccumulator(event_file)
        ea.Reload()
        #scalar_tags = ea.Tags().get('scalars', [])
        # List available tags (scalars, histograms, images, etc.)
        # Read scalar values (e.g., 'loss', 'accuracy')
        try:
            #demo_events = ea.Scalars("JS Divergence")
            demo_events = ea.Scalars("l2 norm demo diff")
            prediction_events = ea.Scalars("Vaccine prediction total") 
        except:
            print("error: ", event_file)
            continue
        #summ = 0
        #for iter, event in enumerate(prediction_events):
        #    print(event, l2_events[iter])
        cur_vac_pred = []
        cur_l2_demo = []
        for event in demo_events:
            cur_l2_demo.append(event.value)
        for event in prediction_events:
            cur_vac_pred.append(event.value)
        all_l2_demographics.append(cur_l2_demo)
        all_vac_predictions.append(cur_vac_pred)

    all_l2_demographics = np.array(all_l2_demographics)
    all_vac_predictions = np.array(all_vac_predictions)

    starting_prediction = []
    min_prediction = []
    for row in range(all_vac_predictions.shape[0]):
        starting_prediction.append(all_vac_predictions[row,0])
        min_prediction.append(np.min(all_vac_predictions[row,:]))

    combined = list(zip(all_names, starting_prediction, min_prediction))

    # Sort by the first element of each tuple (i.e., values from A)
    combined.sort(key=lambda x: x[2])

    # Unzip back into separate lists
    names_sorted, starting_sorted, min_sorted = zip(*combined)

    # Convert back to lists if needed
    names_sorted = list(names_sorted)
    starting_sorted = list(starting_sorted)
    min_sorted = list(min_sorted)
    
    results_dir = {}
    for ii, name_tup in enumerate(names_sorted):
        for nt in name_tup:
            if nt not in results_dir:
                results_dir[nt] = [min_sorted[ii]]
            else:
                results_dir[nt].append(min_sorted[ii])
    print("Mean min: ", np.mean(min_sorted))
    print("")
    print("Lower than mean:")
    for var in results_dir:
        if np.mean(results_dir[var]) < np.mean(min_sorted):
            print(var, np.mean(results_dir[var]))
    print("")

    print("Higher than mean:")
    for var in results_dir:
        if np.mean(results_dir[var]) > np.mean(min_sorted):
            print(var, np.mean(results_dir[var]))

    print("")

    x = 5
    for ii in range(x):
        print(names_sorted[ii], min_sorted[ii])

for runs_path in file_paths:
    print_stats(runs_path)

In [ ]:
#attempting to find trends in the randomness
#plotting events from saved run logs
# Path to the directory where SummaryWriter saved logs
file_paths = ["runs/"]

def print_stats(runs_path):
    run_folders = []
    for name in os.listdir(runs_path):
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))

    all_vac_predictions = []
    all_l2_demographics = []
    for i, event_file in enumerate(event_files):
        # Load event accumulator
        ea = event_accumulator.EventAccumulator(event_file)
        ea.Reload()

        # List available tags (scalars, histograms, images, etc.)
        # Read scalar values (e.g., 'loss', 'accuracy')
        prediction_events = ea.Scalars("Vaccine prediction")
        l2_events = ea.Scalars("L2 Demographics")
        #summ = 0
        #for iter, event in enumerate(prediction_events):
        #    print(event, l2_events[iter])
        cur_vac_pred = []
        cur_l2_demo = []
        for event in prediction_events:
            cur_vac_pred.append(event.value)
        for ievent in l2_events:
            cur_l2_demo.append(ievent.value)
        all_vac_predictions.append(cur_vac_pred)
        all_l2_demographics.append(cur_l2_demo)

    
    all_vac_predictions = np.array(all_vac_predictions)
    all_l2_demographics = np.array(all_l2_demographics)

    r_vales = []
    for i in range(1,70):
        start_point = 80
        end_point = np.shape(all_vac_predictions)[1]-i
        #ending points
        ending_vac_pred = all_vac_predictions[:,end_point]
        ending_l2_demopgrahics = all_l2_demographics[:,end_point]

        ending_vac_pred=np.delete(ending_vac_pred, ending_l2_demopgrahics.argmax())
        ending_l2_demopgrahics=np.delete(ending_l2_demopgrahics, ending_l2_demopgrahics.argmax())

        #plt.scatter(x=ending_l2_demopgrahics, y=ending_vac_pred)
        #plt.xlabel("Ending Demographic L2")
        #plt.ylabel("Ending Vac Prediction")
        #plt.show()
        slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(ending_l2_demopgrahics, ending_vac_pred)
        r_vales.append(r_value)
        #print(r_value)
        #print("R value: ", r_value)
        #print("---------------------------------")
        #average from starting point
        #avg_vac_pred = np.mean(all_vac_predictions[:,start_point:],axis=1)
        #avg_l2_demo = np.mean(all_l2_demographics[:,start_point:],axis=1)
        #print(avg_vac_pred)
        #plt.scatter(x=avg_l2_demo, y=avg_vac_pred)
        #plt.xlabel("avg Demographic L2")
        #plt.ylabel("avg Vac Prediction")
        #slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(avg_vac_pred, avg_l2_demo)
        #print("R value: ", r_value)

    plt.plot(r_vales)
for runs_path in file_paths:
    print_stats(runs_path)

In [ ]:
# HHP Analysis Cell
from analysis import *

indices = entropy_pred_relationship(runs_path,
                                filter_by=["Week=28"], 
                                    num_runs=20,)

all_outs = HHP_load_all_weeks_as_dict('./runs/')

process_all_weeks_dict(all_outs,
                           "",
                           save=False,)

In [ ]:
#creating bias xbox and gt
GT_SIZE = 10000
BIAS_SIZE = 5000

test = XBoxDatasetSimulation("./data/gcHouse,7attributes.csv")
def save_new_XBOX_csvs():
    global_GT_dict, global_GT_var_order,global_bias_dict,global_bias_var_order = XBOX_get_GT_and_bias_ratios()

    GT_persons_count = get_all_persons_types_count(GT_SIZE,global_GT_dict)
    BT_persons_count = get_all_persons_types_count(BIAS_SIZE,global_bias_dict)

    GT_sampled_df = XBOX_get_sampled_df(global_GT_var_order,
                                        GT_persons_count,
                                        test.df)
    bias_sampled_df = XBOX_get_sampled_df(global_bias_var_order,
                                        BT_persons_count,
                                        test.df)

    bias_df_normalized, bias_NaN_columns = normalize_df(bias_sampled_df)
    gt_df_normalized, gt_df_NaN_columns = normalize_df(GT_sampled_df)
    all_NaNs_columns = bias_NaN_columns and gt_df_NaN_columns

    clean_NaN_by_col_index(bias_df_normalized,all_NaNs_columns)
    clean_NaN_by_col_index(gt_df_normalized,all_NaNs_columns)

    print(bias_df_normalized.shape, gt_df_normalized.shape)
    #save to CSV files
    bias_df_normalized = bias_df_normalized.sample(frac=1).reset_index(drop=True)
    gt_df_normalized = gt_df_normalized.sample(frac=1).reset_index(drop=True)

    bias_df_normalized.to_csv(BIAS_SAVE_PATH + "XBOX_bias.csv",index=False)
    gt_df_normalized.to_csv(GT_SAVE_PATH + "XBOX_GT.csv",index=False)

save_new_XBOX_csvs()

In [ ]:
#data analysis:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp

def analyze_dataset_differences(A, B, feature_names=None):
    """
    Analyzes differences between two datasets A and B.

    Args:
        A (np.ndarray): Dataset A, shape (N, D)
        B (np.ndarray): Dataset B, shape (M, D)
        feature_names (list[str], optional): List of feature names for plotting

    Returns:
        dict: Dictionary of comparison metrics per feature
    """
    assert A.shape[1] == B.shape[1], "Datasets must have the same number of features"
    D = A.shape[1]
    results = {}

    if feature_names is None:
        feature_names = [f"Feature {i}" for i in range(D)]

    for i in range(D):
        feat_A = A[:, i]
        feat_B = B[:, i]
        
        # Basic statistics
        mean_A, std_A = np.mean(feat_A), np.std(feat_A)
        mean_B, std_B = np.mean(feat_B), np.std(feat_B)
        
        # KS-test for distributional difference
        ks_stat, ks_pval = ks_2samp(feat_A, feat_B)

        results[feature_names[i]] = {
            "mean_A": mean_A,
            "mean_B": mean_B,
            "std_A": std_A,
            "std_B": std_B,
            "ks_stat": ks_stat,
            "ks_pval": ks_pval,
        }

        # Plot distributions
        plt.figure(figsize=(6, 4))
        sns.kdeplot(feat_A, label='Dataset A', fill=True)
        sns.kdeplot(feat_B, label='Dataset B', fill=True)
        plt.title(f"Distribution of {feature_names[i]}")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return results
GT_LIMIT = 25000
BIAS_LIMIT = 1000
seed=0
rngs = set_seed(seed,
                device,
                data_init=[True,True],
                data_gen =True,
                network_init=True,)
week='29'
D1_rngs = copy.deepcopy(rngs)
D1_rngs['seed_bias'] = 359556
D2_rngs = copy.deepcopy(rngs)
D2_rngs['seed_bias'] = 280651
#load in both data sets
D1 = HouseholdPulse_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
                            bias_path = './data/censusHouseholdPulse_data/pulse_week'+week+'_cleaned.csv',
                            rngs=D1_rngs,
                            device=device,
                            gt_limit = GT_LIMIT,
                            bias_limit = BIAS_LIMIT, 
                            )
D2 = HouseholdPulse_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
                            bias_path = './data/censusHouseholdPulse_data/pulse_week'+week+'_cleaned.csv',
                            rngs=D2_rngs,
                            device=device,
                            gt_limit = GT_LIMIT,
                            bias_limit = BIAS_LIMIT, 
                            )

analyze_dataset_differences(D1.unscaled_biased, D2.unscaled_biased, feature_names=None)

In [ ]:
#deepset classifier helper functions
def sample_dataset(dataset, batch_size, sample_size, rngs, weights=None):
    if weights is None:
        index = np.random.choice(np.arange(len(dataset)), size=sample_size*batch_size, p=weights)
    else:
        index = rngs['np'].choice(np.arange(len(dataset)),size=sample_size*batch_size)
    sampled_data = torch.clone(dataset[index])
    sampled_data = torch.reshape(sampled_data, (batch_size, sample_size, dataset.shape[1]))
    return sampled_data

def binary_accuracy_from_logits(logits, targets, threshold=0.5):
    preds = (logits > threshold).float()
    correct = (preds == targets).sum().item()
    total = targets.size(0)
    return 100.0 * correct / total


In [ ]:
#making new household census data

from HouseholdCensusDataProcessing import * 
census_df = None
survey_df = None

for w in range(29,30):
    print("processing: ", w)
    week = str(w)
    #week="ALL"
    #census_df = pd.read_csv("./data/censusHouseholdPulse_data/raw_census/usa_00008.csv")
    #survey_df = pd.read_csv("./data/censusHouseholdPulse_data/raw_survey/pulse2021_puf_"+week+".csv")
    survey_df = load_evenly_sampled_csv_rows("./data/censusHouseholdPulse_data/raw_survey/", 2500, "HLTHINS1")
    survey_df, census_df = recoding_survey_and_census_data(survey_df, census_df, target_var=['RECVDVACC'])
    #survey_df.to_csv('./data/censusHouseholdPulse_data/cleaned/pulse_week'+week+'_cleaned.csv', index=False)
    #census_df.to_csv('./data/censusHouseholdPulse_data/cleaned/ipums_cleaned.csv',index=False)

In [ ]:
#compare dataframes
def compare_dataframes(df1, df2):
    assert list(df1.columns) == list(df2.columns), "Columns must match"

    cols = df1.columns
    n_cols = len(cols)
    n_rows = (n_cols + 2) // 3  # auto-layout: 3 columns per row

    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
    axes = axes.flatten()

    for i, col in enumerate(cols):
        ax = axes[i]
        ax.boxplot([df1[col].dropna(), df2[col].dropna()], labels=["Survey", "Census"])
        ax.set_title(f"Column: {col}")
        ax.grid(True)

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

compare_dataframes(survey_df.iloc[:, 1:],census_df.iloc[:,1:])

In [ ]:
def sample_categorical_distribution(df, column, target_dist, K, replace=False, random_state=None):
    """
    Sample K rows from df such that the distribution of the values in the specified column
    matches the target categorical distribution.

    Parameters:
        df (pd.DataFrame): Original DataFrame.
        column_index (int): Index of the column to match distribution on.
        target_dist (dict): Target distribution (e.g., {0: 0.5, 1: 0.3, 2: 0.2}).
        K (int): Total number of samples to draw.
        replace (bool): Whether to sample with replacement.
        random_state (int or None): Seed for reproducibility.

    Returns:
        pd.DataFrame: Sampled DataFrame of size K.
    """
    np.random.seed(random_state)
    result_dfs = []
    
    for category, proportion in target_dist.items():
        num_samples = int(round(proportion * K))
        subset = df[df[column] == category]
        
        if len(subset) == 0:
            raise ValueError(f"No samples found for category '{category}' in the specified column.")
        if not replace and num_samples > len(subset):
            raise ValueError(f"Not enough samples in category '{category}' to sample {num_samples} without replacement.")
        
        sampled = subset.sample(n=num_samples, replace=replace, random_state=random_state)
        result_dfs.append(sampled)
    
    result = pd.concat(result_dfs).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    # Fix any rounding issues (e.g., if total != K due to rounding)
    if len(result) > K:
        result = result.sample(n=K, random_state=random_state)
    elif len(result) < K:
        extra = df.sample(n=K - len(result), replace=replace, random_state=random_state)
        result = pd.concat([result, extra]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    return result

def solo_var_experiment(ground_truth_path,
                        target_column,
                        target_dist,
                        gt_size,
                        bias_size,):
    raw_gt_load = pd.read_csv(ground_truth_path)

    biased_dataset = sample_categorical_distribution(raw_gt_load, target_column, target_dist, bias_size, replace=False, random_state=None).to_numpy(dtype=np.float, na_value=0)
    gt_dataset = raw_gt_load.sample(n=gt_size).to_numpy(dtype=np.float, na_value=0)
    

ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv'

In [ ]:
%autoreload 2
#making new ari data set
from AriSurveyDataProcessing import * 
census_df = None
survey_df = None

census_df = pd.read_csv("./data/ari_survey/usa_00009.csv")
survey_df = pd.read_csv("./data/ari_survey/aug23data.csv")
#survey_df, census_df = recoding_survey_and_census_data(survey_df, census_df)
#survey_df.to_csv('./data/ari_survey/aricleaned.csv', index=False)
#census_df.to_csv('./data/ari_survey/ipums_cleaned.csv',index=False)

In [ ]:
survey_df['age']

In [ ]:
accepted_answers = ['Full-time', 'Retired', 'Part-time', 'Unemployed', 'Homemaker', 'Permanently disabled',
                            'Student', 'Other, please specify:', 'Temporarily laid off']
survey_df[survey_df['employmentstatus'].isin(accepted_answers)]

Transform raw data table into discretized/binned data 
1. Remove irrelevant columns
    a. List out available columns - "list(df.columns)"
    b. pick out variables that have an IPUMS equivalent
    c. Remove irrelevant columns - df = df[relevant_columns] (relevant_columns: list) 
    d. or call df = df.filter(items=relevant_survey_col)
2. "fix" (remove) NaN entries in relevant columns
    a. check responses - df[var_name].value_counts()
    b. create a list of "accepted_answers"
    c. use accepted answers to create a "mask" - df[var_name].isin(accepted_answers)
    d. use mask to remove bad columns - df = df[bad_answer_mask]
    e. Next - check for na entries. create a mask - df[var_name].isna()
    f. remove NaN entries df = df[na_mask]
3. Bin relevant columns
    a. create local function that takes "value" as input and returns the appropriate bin
    b. call df[var_name].apply(binning_function)
    c. Store df's as CSV files
4. Additional but important modifcations for the NN. 
    a. The features/variables for survey/census appear in the same order. 
        - in (1) when you create list of relevant columns. calling df[relevant_columns], will automatically reorder into the same order
    b. What happens if variable names are not the same? e.g. age vs AGE
        - df.rename(columns={old_name:new_name},inplace=True)
        - I renamed all the survey columns to the census name

Now we have the data set made

How to call the actual function - creating the data set object
1. One column must exist in a specific location in the survey data set - column 0 must be the labels
2. one column must exist in a specific location in the census data set - column 0 must be the PERWT IPUMS variable
3. Create class new_dataset(HouseholdPulse_dataset) that inherits household pulse object. 
    a. This object takes:
        - file path - str - to survey dataframe
        - file path - str - to census data frame
        - GT_LIMIT - int - number of census points to use
        - BIAS_LIMIT - int - number of survey points to use

How to call the actual function - GAN object
1. create a WGAN_GP object that takes as input:
    a. data set (created above) object
    b. Tuning parameters:
        - generator_type='deepSet' - generator architure. default - deepSet; no real need to change
        - discriminator_type='deepSet' - critic/discriminator architure. default - deepSet; no real need to change
        - gen_learning_rate=hparams["glearningrate"] - float - controls how fast generator converges. might need tuning
        - disc_learning_rate=hparams["dlearningrate"] - float - controls how fast disc converges. might need tuning 
        - batch_size=hparams["batch_size"] - int - can keep at default (16), how many "subsets" of data points the discriminator sees at a time
        - subset_size - int - default (128) how many data points on which the discriminator makes a decision. 
        - gen_layers=hparams["gen_layers"] - list[int] - size of the gen network, default size is 1 hidden layer of 1024. This will have minor impact
        - disc_layers=hparams["disc_layers"] - list[int] - size of the disc network, default size is 1 hidden layer of 1024
        - lambda_gp=hparams["lambdagp"] - float - default - keep for theoretical guaruntees 
        - lambda_weights=hparams["lambdaw"] - float - default - punishes high concentration of probabilities into a small number of points
        - lambda_demo=hparams["lambdad"] - float - default - punishes high deviation from census demographic 
        - temperature=hparams["tau"] - float - default - needed for generator
        - generator_dropout=hparams["generator_dropout"] - float - default - prevents overfitting
        - discriminator_dropout=hparams["discriminator_dropout"] - float - default - prevents overfitting

Measuring demographic similarity between a weighted survey and a census data set:
1. Currently - treat each variable independently
    a. For each variable, measure its JSD similarity to the census (histogram comparison)
    b. average the histogram similarity score
    c. similar to raking - treat variables independently, how closely can i match each of the distributions
2. Alternative - treat all variables simultaneously using the intersection of individuals 
    a. caveat - whent here are a lot of variables, the intersected space gets extremely large (e.g. 5 bins with 9 variables, thats 5^9 unique individuals)
        - there are a lot of combinations in the census that wont exist in the survey, and vice versa
    b. multi-level modeling post stratification - does groups of variables at a time, rather than the full span of variables
    c. Even if we trained each var independently, does it correct for the skew of specific interactions
    d. Synthetic experiments
        - Upscale a specific interaction of 2 - 3 variables (start with 2)
        - Train system using default method - use all the variables (10 variables)
        - Meausre the recovery (or lack thereof) of the upscaled interaction - measuring the balance of all interactions of the 2-3 variables

In [ ]:
educ_mapping = {
            'Less than high school': 1,  # N/A or no schooling
            'Some high school': 1,
            'High school graduate or equivalent (for example GED)': 2,  # High school graduate
            'Some college, but degree not received or is in progress': 3,  # Some college
            'Associate\'s degree (for example AA, AS)': 3,  # Some college
            'Bachelor\'s degree (for example BA, BS, AB)': 4, # Bachelor's degree
            'Graduate degree (for example master\'s, professional, doctorate)': 5  # Graduate degree
        }

survey_df['education'] = survey_df['education'].map(educ_mapping)
survey_df['education'].value_counts()

In [ ]:
survey_df['education'].value_counts()

In [ ]:
census_df['AGE'].value_counts()

In [ ]:
survey_df_orig['employmentstatus'].value_counts()